# CTGAN on the Adult Dataset

This notebook implements **CTGAN** on the **Adult** dataset from the UCI Machine Learning Repository.

## Data loading

(Note: in this notebook, the Adult dataset is imported directly from `ucimlrepo`, which returns the full dataset and separates it into features and target. In the earlier version, the data came from a pre-separated training split. Therefore, the import procedure is slightly different, but the dataset content and the overall EDA conclusions should remain essentially the same.)

In [3]:
from ucimlrepo import fetch_ucirepo 
import pandas as pd

# fetch dataset 
adult = fetch_ucirepo(id=2) 
  
# data (as pandas dataframes) 
X = adult.data.features
y = adult.data.targets

df = pd.concat([X, y], axis=1)

In [4]:
df.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [5]:
for col in df.select_dtypes(include='object').columns:
    bad = (df[col] != df[col].astype(str).str.strip()).sum()
    if bad > 0:
        print(col, bad)

workclass 963
occupation 966
native-country 274


The object columns contain leading spaces in some rows, so they are stripped before further preprocessing.

In [6]:
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip()

The `?` entries are treated as missing values and then filled with the category `Missing` in the categorical columns so that no rows need to be dropped at this stage.

In [7]:
df = df.replace('?', pd.NA)
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].fillna('Missing')

In [8]:
print(df.duplicated().sum())

29


A small number of duplicate rows is present. These are kept, since identical records may correspond to different individuals with the same recorded attributes.

## Column groups

The columns are separated into categorical and numerical features. For CTGAN, the categorical columns are passed as `discrete_columns`, which is the parameter name expected by the library.

In [9]:
categorical_columns = [
    'workclass', 'education', 'marital-status', 'occupation',
    'relationship', 'race', 'sex', 'native-country', 'income'
]

numerical_columns = [
    'age', 'fnlwgt', 'education-num',
    'capital-gain', 'capital-loss', 'hours-per-week'
]

discrete_columns = categorical_columns

In [10]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df['income']
)

## Train-test split

The dataset is split into **80% training data** and **20% test data**.  
`random_state=42` ensures reproducibility, and `stratify=df['income']` preserves the class distribution of the target in both splits.

## Quick check

Before training, it is useful to confirm the split sizes, verify that no missing values remain in the categorical columns, and check the class balance in the target column.

In [11]:
print('Train shape:', train_df.shape)
print('Test shape:', test_df.shape)
print('\nRemaining missing values:')
print(df.isna().sum()[df.isna().sum() > 0])
print('\nTarget distribution in full data:')
print(df['income'].value_counts(normalize=True))
print('\nTarget distribution in train data:')
print(train_df['income'].value_counts(normalize=True))
print('\nTarget distribution in test data:')
print(test_df['income'].value_counts(normalize=True))

Train shape: (39073, 15)
Test shape: (9769, 15)

Remaining missing values:
Series([], dtype: int64)

Target distribution in full data:
income
<=50K     0.506122
<=50K.    0.254596
>50K      0.160538
>50K.     0.078744
Name: proportion, dtype: float64

Target distribution in train data:
income
<=50K     0.506104
<=50K.    0.254600
>50K      0.160546
>50K.     0.078750
Name: proportion, dtype: float64

Target distribution in test data:
income
<=50K     0.506193
<=50K.    0.254581
>50K      0.160508
>50K.     0.078718
Name: proportion, dtype: float64


## Summary

So far, the notebook:
- loads the Adult dataset from `ucimlrepo`
- merges features and target into one DataFrame
- removes hidden leading and trailing spaces from categorical values
- converts `?` markers to missing values
- fills missing categorical values with `Missing`
- keeps duplicate rows
- defines categorical and numerical column groups
- creates a stratified 80/20 train-test split

The data is now ready for CTGAN training.

In [12]:
from ctgan import CTGAN